In [0]:
display(dbutils.fs.ls("abfss://azuredata@azure00014.dfs.core.windows.net/"))

In [0]:
display(dbutils.fs.ls("abfss://azuredata@azure00014.dfs.core.windows.net/raw/"))

In [0]:
raw_path = "abfss://azuredata@azure00014.dfs.core.windows.net/raw/"

first_item = dbutils.fs.ls(raw_path)[0].path

display(dbutils.fs.ls(first_item))

In [0]:
display(dbutils.fs.ls(first_item))

In [0]:
second_item = dbutils.fs.ls(first_item)[0].path

display(dbutils.fs.ls(second_item))

In [0]:
raw_path = "abfss://azuredata@azure00014.dfs.core.windows.net/raw/"

df_raw = (
    spark.read
    .option("recursiveFileLookup", "true")
    .json(raw_path)
)

display(df_raw)

In [0]:
df_raw.printSchema()

In [0]:
from pyspark.sql.functions import explode, col

df_ingested = (
    df_raw
    .select(explode(col("MRData.RaceTable.Races")).alias("race"))
    .select(
        col("race.season").alias("season"),
        col("race.round").alias("round"),
        col("race.raceName").alias("raceName"),
        col("race.date").alias("date"),
        col("race.Circuit.circuitId").alias("circuitId"),
        col("race.Circuit.circuitName").alias("circuitName"),
        col("race.Circuit.Location.country").alias("country"),
        col("race.Circuit.Location.locality").alias("locality"),
        col("race.Circuit.Location.lat").alias("lat"),
        col("race.Circuit.Location.long").alias("long")
    )
)

display(df_ingested)

In [0]:
ingested_path = "abfss://azuredata@azure00014.dfs.core.windows.net/ingested/"

(
    df_ingested
    .write
    .mode("overwrite")
    .parquet(ingested_path)
)

print("Ingested data written successfully.")

In [0]:
display(dbutils.fs.ls(ingested_path))